# Phase 4: Self-Supervised Learning via SimCLR (PlantVillage)

- **Q4.11** — InfoNCE Loss: Train ResNet18 encoder + MLP projection head with contrastive loss only (ignore labels)  
- **Q4.12** — Linear Evaluation Protocol: Freeze encoder, train a single `nn.Linear` head with labels, compare test accuracy to the supervised CNN from Assignment 2

Dataset split: random 70 / 15 / 15 (`torch.Generator().manual_seed(42)`) — same as Assignment 2.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Dataset

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


---
## Dataset — PlantVillage (color), same split as Assignment 2

In [2]:
DATA_DIR = '../../Assignment_2/Datasets/plantvillage_dataset/color'

# Standard eval transform — used for linear eval training/evaluation
eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = torchvision.datasets.ImageFolder(root=DATA_DIR, transform=eval_transform)
NUM_CLASSES  = len(full_dataset.classes)
N = len(full_dataset)

n_train = int(0.70 * N)
n_val   = int(0.15 * N)
n_test  = N - n_train - n_val

gen = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [n_train, n_val, n_test], generator=gen
)
print(f'Classes: {NUM_CLASSES} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

Classes: 38 | Train: 38013 | Val: 8145 | Test: 8147


---
## Q4.11 — InfoNCE Loss & SimCLR Training

SimCLR requires **two independently augmented views** of the same image as a positive pair.
`TwoViewDataset` wraps the train subset so each `__getitem__` returns `(view1, view2, label)` —
labels are **ignored** during contrastive pre-training.

In [3]:
# ── SimCLR augmentation pipeline (random crops, flips, color jitter) ─────────
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class TwoViewDataset(Dataset):
    """Returns two independently augmented views of the same image."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
        self.base      = subset.dataset   # underlying ImageFolder

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        real_idx = self.subset.indices[idx]
        path, label = self.base.samples[real_idx]
        img = self.base.loader(path)
        return self.transform(img), self.transform(img), label


simclr_loader = DataLoader(
    TwoViewDataset(train_dataset, simclr_transform),
    batch_size=256, shuffle=True, num_workers=0, drop_last=True
)

# Single-view loaders for linear eval and final accuracy
val_loader  = DataLoader(val_dataset,  batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

In [4]:
# ── SimCLR model: ResNet18 encoder + 2-layer MLP projection head ─────────────
class SimCLRModel(nn.Module):
    """ResNet18 backbone (fc replaced with Identity) + MLP projection head."""
    def __init__(self, proj_dim=128):
        super().__init__()
        backbone = resnet18(weights=None)
        feat_dim = backbone.fc.in_features   # 512
        backbone.fc = nn.Identity()          # strip classification head
        self.encoder   = backbone
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim),
        )

    def forward(self, x):
        h = self.encoder(x)     # (B, 512) — used for linear eval
        z = self.projector(h)   # (B, proj_dim) — used for contrastive loss
        return h, z


# ── NT-Xent / InfoNCE loss implemented from scratch ──────────────────────────
def nt_xent_loss(z1, z2, temperature=0.5):
    """Normalized Temperature-scaled Cross-Entropy loss (InfoNCE).

    Given N image pairs, concatenate to 2N vectors.
    Positive for row i: column i+N (and vice-versa).
    All other 2N-2 entries in the row are negatives.
    """
    N = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)  # (2N, D), L2-normalised

    sim = torch.mm(z, z.T) / temperature                 # (2N, 2N) cosine similarities

    # Remove self-similarity from denominator
    sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=z.device), float('-inf'))

    # Positive indices: row i -> column i+N; row i+N -> column i
    labels = torch.cat([torch.arange(N, 2*N), torch.arange(N)]).to(z.device)

    return F.cross_entropy(sim, labels)


print('SimCLRModel and NT-Xent loss defined.')

SimCLRModel and NT-Xent loss defined.


In [ ]:
# ── Train SimCLR with contrastive loss only (no labels) ───────────────────────
EPOCHS_SIMCLR = 30
LR_SIMCLR     = 3e-4
TEMPERATURE   = 0.5

simclr_model = SimCLRModel(proj_dim=128).to(device)
optimizer    = optim.Adam(simclr_model.parameters(), lr=LR_SIMCLR, weight_decay=1e-4)

train_losses = []
for epoch in range(1, EPOCHS_SIMCLR + 1):
    simclr_model.train()
    total_loss = 0.0
    for v1, v2, _ in simclr_loader:        # labels (_) ignored
        v1, v2 = v1.to(device), v2.to(device)
        _, z1  = simclr_model(v1)
        _, z2  = simclr_model(v2)
        loss   = nt_xent_loss(z1, z2, TEMPERATURE)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(simclr_loader)
    train_losses.append(avg)
    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS_SIMCLR}  train NT-Xent: {avg:.4f}')

# ── Report final NT-Xent loss on validation set ───────────────────────────────
simclr_model.eval()
val_2v_loader = DataLoader(
    TwoViewDataset(val_dataset, simclr_transform),
    batch_size=256, shuffle=False, num_workers=0, drop_last=False
)
val_loss_total = 0.0
with torch.no_grad():
    for v1, v2, _ in val_2v_loader:
        v1, v2 = v1.to(device), v2.to(device)
        _, z1  = simclr_model(v1)
        _, z2  = simclr_model(v2)
        val_loss_total += nt_xent_loss(z1, z2, TEMPERATURE).item()
val_loss = val_loss_total / len(val_2v_loader)
print(f'\nFinal NT-Xent loss on validation set: {val_loss:.4f}')

Epoch   5/30  train NT-Xent: 4.6684
Epoch  10/30  train NT-Xent: 4.5657
Epoch  15/30  train NT-Xent: 4.5158
Epoch  20/30  train NT-Xent: 4.4906


In [ ]:
# ── Training loss curve ────────────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS_SIMCLR + 1), train_losses, lw=2)
plt.xlabel('Epoch')
plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Training Loss (ResNet18 + MLP Projection Head)')
plt.grid(True)
plt.tight_layout()
plt.show()

---
## Q4.12 — Linear Evaluation Protocol

Freeze all SimCLR encoder weights.  
Train a single **untrained** `nn.Linear(512, NUM_CLASSES)` head using true PlantVillage labels.  
Compare test accuracy to the fully supervised CNN from Assignment 2.

In [ ]:
# ── Freeze encoder weights ────────────────────────────────────────────────────
for p in simclr_model.encoder.parameters():
    p.requires_grad = False
simclr_model.eval()   # keeps BatchNorm in eval mode (uses running stats)

# ── Single untrained linear head ──────────────────────────────────────────────
linear_head   = nn.Linear(512, NUM_CLASSES).to(device)
lin_optimizer = optim.Adam(linear_head.parameters(), lr=1e-3)
criterion     = nn.CrossEntropyLoss()

EPOCHS_LINEAR    = 20
train_lin_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)

for epoch in range(1, EPOCHS_LINEAR + 1):
    linear_head.train()
    total_loss = 0.0
    for imgs, labels in train_lin_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            h, _ = simclr_model(imgs)   # frozen 512-d representation
        loss = criterion(linear_head(h), labels)
        lin_optimizer.zero_grad()
        loss.backward()
        lin_optimizer.step()
        total_loss += loss.item()
    if epoch % 5 == 0:
        print(f'Linear eval epoch {epoch:2d}/{EPOCHS_LINEAR}  loss: {total_loss/len(train_lin_loader):.4f}')

In [ ]:
# ── SimCLR linear eval accuracy ───────────────────────────────────────────────
def accuracy(enc, head, loader):
    """Accuracy of frozen encoder + linear head on a DataLoader."""
    enc.eval(); head.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = head(enc(imgs)[0]).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

val_acc  = accuracy(simclr_model, linear_head, val_loader)
test_acc = accuracy(simclr_model, linear_head, test_loader)
print(f'SimCLR Linear Eval — Val  accuracy : {val_acc*100:.2f}%')
print(f'SimCLR Linear Eval — Test accuracy : {test_acc*100:.2f}%')

In [ ]:
# ── Load + evaluate Assignment 2 supervised CNN for comparison ────────────────
class PlantCNN(nn.Module):
    """3-block CNN from Assignment 2 Q2 — same architecture as best_model_q2_4.pt."""
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  8, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(8);  self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(16); self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(16,32, 3, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(32); self.pool3 = nn.MaxPool2d(2)
        self.relu = nn.ReLU(inplace=True); self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32*16*16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        return self.fc2(self.relu(self.fc1(self.flatten(x))))


MODEL_PATH = '../../Assignment_2/src/best_model_q2_4.pt'
cnn = PlantCNN(num_classes=NUM_CLASSES).to(device)
cnn.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=False))
cnn.eval()

cnn_correct = cnn_total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = cnn(imgs).argmax(dim=1)
        cnn_correct += (preds == labels).sum().item()
        cnn_total   += labels.size(0)
supervised_cnn_test_acc = cnn_correct / cnn_total

print(f'Supervised CNN (A2) — Test accuracy : {supervised_cnn_test_acc*100:.2f}%')
print(f'Gap (supervised - SimCLR)           : {(supervised_cnn_test_acc - test_acc)*100:.2f}pp')

In [ ]:
# ── Bar chart: SimCLR Linear Eval vs Supervised CNN ───────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ['SimCLR\nLinear Eval', 'Supervised CNN\n(Assignment 2)'],
    [test_acc*100, supervised_cnn_test_acc*100],
    color=['steelblue', 'darkorange'], width=0.4
)
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Self-Supervised (SimCLR) vs Supervised CNN')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()